In [1]:
from flax import linen as nn
import jax 
import jax.numpy as jnp

from liquid_solver import LEsolver


solver = LEsolver()

import jax 
import jax.numpy as jnp
from flax import linen as nn

def get_layers(neurons: tuple[int, ...]):

    if not neurons:
        return None
    
    layers = []
    for n in neurons:
        layers.append(
            nn.Dense(n)
        )
    return tuple(layers)

def forward(h, last_linear, layers):
    if not layers: 
        return h
    
    for layer in layers[:-1]:
        h = nn.relu(layer(h))
    
    final_layer = layers[-1]
    h = final_layer(h)
    
    return h if last_linear else nn.relu(h)   

class LeModelMlp(nn.Module):
    
    out: tuple[int, ...]
    delegation: tuple[int, ...]
    body: tuple[int, ...] | None = None
    
    def setup(self):
        self.body_layers = get_layers(self.body)
        self.out_layers = get_layers(self.out)
        self.delegation_layers = get_layers(self.delegation)

    def __call__(self, x: jax.Array):
        h = x
        h_body = forward(h, last_linear=False, layers=self.body_layers)
        y = forward(h_body, last_linear=True, layers=self.out_layers)
        d = forward(h_body, last_linear=True, layers=self.delegation_layers)

        return y, d
    

class LeMlp(nn.Module):
    n_models: int
    out: tuple[int, ...]
    delegation: tuple[int, ...]
    body: tuple[int, ...] | None = None

    def setup(self):
        
        assert self.delegation[-1] == self.n_models, "Last dim of delegation needs to equal to n_models"

        VmappedLeModelMlp = nn.vmap(
            LeModelMlp,
            variable_axes={'params': 0},
            split_rngs={'params': True}, # Vmap over different models
            in_axes=None, # Do not vmap over batch elements
            axis_size=self.n_models,
            out_axes=1 # Stack the model outputs to axis=1
        )
        
        self.ensemble = VmappedLeModelMlp(
            out=self.out, 
            delegation=self.delegation, 
            body=self.body
        )

    def __call__(self, x: jax.Array):
        x = x if x.ndim == 2 else x.reshape((x.shape[0], -1))
        ys, d = self.ensemble(x)
        d = nn.softmax(d, axis=-1)
        leinfo = solver.solve_power(d)
        y = solver.mix_power_mean(ys, leinfo.power)
        return y

def make_model_le(width: int, n_models: int, out: int = 1):
    
    return LeMlp(
        n_models=n_models,
        body=(width,),
        out=(out,),
        delegation=(n_models,)
    )

class MLP(nn.Module):
    h: int
    out_dim: int = 1
    @nn.compact
    def __call__(self, x):
        x = nn.Dense(self.h)(x)
        x = nn.relu(x)
        x = nn.Dense(self.out_dim)(x)
        return x

def make_model_mlp(width: int, n_models: int, out: int = 1):
    
    P1 = (width + width + width + 1 + (width * n_models) + n_models) * n_models
    aw = (P1 - 1) / 3
    aw = int(aw)

    return MLP(
        h=aw,
        out_dim=out
    )

class DeModelMlp(nn.Module):
    
    out: tuple[int, ...]
    body: tuple[int, ...] | None = None
    
    def setup(self):
        self.body_layers = get_layers(self.body)
        self.out_layers = get_layers(self.out)

    def __call__(self, x: jax.Array):
        h = x
        h_body = forward(h, last_linear=False, layers=self.body_layers)
        y = forward(h_body, last_linear=True, layers=self.out_layers)
        return y

class DeMlp(nn.Module):
    n_models: int
    out: tuple[int, ...]
    body: tuple[int, ...] | None = None

    def setup(self):
    

        VmappedDeModelMlp = nn.vmap(
            DeModelMlp,
            variable_axes={'params': 0},
            split_rngs={'params': True}, # Vmap over different models
            in_axes=None, # Do not vmap over batch elements
            axis_size=self.n_models,
            out_axes=1 # Stack the model outputs to axis=1
        )
        
        self.power = self.param("powerlearn", nn.initializers.ones, (self.n_models,))

        self.ensemble = VmappedDeModelMlp(
            out=self.out,  
            body=self.body
        )

    def __call__(self, x: jax.Array):
        x = x if x.ndim == 2 else x.reshape((x.shape[0], -1))
        ys = self.ensemble(x)
        power = self.power / self.power.sum()
        power = jnp.broadcast_to(power, (x.shape[0], self.n_models))
        y = solver.mix_power_mean(ys, power)
        return y

def make_model_de(width: int, n_models: int, out: int = 1):
    # LE params count
    P1 = (width + width + width + 1 + (width * n_models) + n_models) * n_models
    
    # DE params: n_models * (w_de + w_de + w_de + 1) + n_models (for self.power)
    # P1 = n_models * (3 * w_de + 2)
    w_de = int(((P1 / n_models) - 2) / 3)

    return DeMlp(
        n_models=n_models,
        body=(w_de,),
        out=(out,),
    )

In [8]:
import random

import numpy as np
import ipywidgets as W
from ipycanvas import Canvas, hold_canvas
from IPython.display import display
import jax
import jax.numpy as jnp
import flax.linen as nn
import optax
from collections import defaultdict

import symlog

train_xy, val_xy = [], []
state = {
    "mode":"train","stop":False,"model":None,"steps":[],"vals":[], "trains":[],
    "xlim":[-1.0,1.0],"ylim":[-1.0,1.0],
    "hist_w_val": [], "hist_w_train": [],
    "curves":[], "ckpt_idx":None, "loss_axes":None, "run_labels": []
}

w = W.Text(value="32", description="w")
e = W.Text(value="10", description="experts")
T = W.BoundedIntText(value=5_000, min=1, max=100000, step=1, description="T")
lr = W.BoundedFloatText(value=5e-4, min=1e-6, max=1.0, step=1e-3, description="lr")
upd = W.BoundedIntText(value=25, min=1, max=500, step=1, description="update/steps")
mode = W.ToggleButtons(options=[("train","train"),("val","val")], value="train", description="add")
train_btn = W.Button(description="Train", button_style="success")
stop_btn = W.Button(description="Stop", button_style="warning")
undo_btn = W.Button(description="Undo")
clear_btn = W.Button(description="Clear", button_style="danger")
expr = W.Textarea(value="np.sum(np.cos(np.outer(x, np.random.normal(0, 10, 100)) + np.random.uniform(0, 2*np.pi, 100)), axis=1)", description="expr", layout=W.Layout(width="500px", height="35px"))
npts = W.BoundedIntText(value=45, min=1, max=200000, step=1, description="n")
vfrac = W.BoundedFloatText(value=0.3, min=0.0, max=0.95, step=0.01, description="val%")
expr_btn = W.Button(description="Expression", button_style="info")
task_type = W.ToggleButtons(options=["regression", "classification"], value="regression", description="Task")
arch_type = W.Text(value="le", description="Archs")
n_bins_w = W.BoundedIntText(value=10, min=2, max=100, step=1, description="Bins")
two_way_w = W.Checkbox(value=True, description="Two-way cont")

status = W.HTML("")

plot = Canvas(width=720, height=360)
loss = Canvas(width=520, height=360)
vhist = Canvas(width=520, height=360)
thist = Canvas(width=520, height=360)

pad = 40

def bounds():
    return (*state["xlim"], *state["ylim"])

def maybe_expand_limits(x,y):
    x0,x1 = state["xlim"]
    y0,y1 = state["ylim"]
    changed = False
    if x < x0:
        dx = x1-x0
        state["xlim"][0] = x - 0.15*dx
        changed = True
    if x > x1:
        dx = x1-x0
        state["xlim"][1] = x + 0.15*dx
        changed = True
    if y < y0:
        dy = y1-y0
        state["ylim"][0] = y - 0.15*dy
        changed = True
    if y > y1:
        dy = y1-y0
        state["ylim"][1] = y + 0.15*dy
        changed = True
    return changed

def world_to_pixel(x,y, x0,x1,y0,y1, Wd,Hd):
    px = pad + (x-x0)/(x1-x0)*(Wd-2*pad)
    py = Hd-pad - (y-y0)/(y1-y0)*(Hd-2*pad)
    return px,py

def pixel_to_world(px,py, x0,x1,y0,y1, Wd,Hd):
    x = x0 + (px-pad)/(Wd-2*pad)*(x1-x0)
    y = y0 + (Hd-pad-py)/(Hd-2*pad)*(y1-y0)
    return float(x), float(y)

def in_axes(px,py, canv):
    Wd,Hd = canv.width, canv.height
    return (pad <= px <= Wd-pad) and (pad <= py <= Hd-pad)

def clear_canvas(c):
    c.clear()
    c.fill_style="#fff"
    c.fill_rect(0,0,c.width,c.height)

def draw_frame(c, title, xlab, ylab):
    Wd,Hd = c.width, c.height
    clear_canvas(c)
    c.stroke_style="#000"
    c.line_width=1
    c.stroke_rect(pad,pad,Wd-2*pad,Hd-2*pad)
    c.fill_style="#000"
    c.font="14px sans-serif"
    c.fill_text(title,10,18)
    c.font="12px sans-serif"
    c.fill_text(xlab, Wd//2-10, Hd-10)
    c.save()
    c.translate(14, Hd//2+10)
    c.rotate(-np.pi/2)
    c.fill_text(ylab,0,0)
    c.restore()

def draw_ticks(c, xticks=None, yticks=None):
    Wd,Hd = c.width, c.height
    if xticks is not None:
        c.font="11px sans-serif"
        for v,lab in xticks:
            x = pad + v*(Wd-2*pad)
            c.stroke_style="#000"
            c.begin_path()
            c.move_to(x, Hd-pad)
            c.line_to(x, Hd-pad+5)
            c.stroke()
            c.fill_style="#000"
            c.fill_text(f"{lab:.3f}", x-14, Hd-pad+18)
    if yticks is not None:
        c.font="11px sans-serif"
        for v,lab in yticks:
            y = Hd-pad - v*(Hd-2*pad)
            c.stroke_style="#000"
            c.begin_path()
            c.move_to(pad-5, y)
            c.line_to(pad, y)
            c.stroke()
            c.fill_style="#000"
            c.fill_text(f"{lab:.3f}", 6, y+4)

def add_polyline(c, xs, ys, x0,x1,y0,y1, line_width=2, stroke="#000"):
    Wd,Hd = c.width, c.height
    c.stroke_style=stroke
    c.line_width=line_width
    c.begin_path()
    started = False
    for xx,yy in zip(xs,ys):
        px,py = world_to_pixel(float(xx), float(yy), x0,x1,y0,y1,Wd,Hd)
        if not started:
            c.move_to(px,py)
            started = True
        else:
            c.line_to(px,py)
    c.stroke()

def add_markers(c, pts, x0,x1,y0,y1, marker="circle", r=3, line_width=1.5, color="#000", filled=True):
    Wd,Hd = c.width, c.height
    if marker == "circle":
        c.fill_style = color
        for x,y in pts:
            px,py = world_to_pixel(float(x), float(y), x0,x1,y0,y1,Wd,Hd)
            c.begin_path()
            c.arc(px,py,r,0,2*np.pi)
            if filled:
                c.fill()
            else:
                c.stroke_style=color
                c.line_width=line_width
                c.stroke()
    elif marker == "cross":
        c.stroke_style = color
        c.line_width = line_width
        s = float(r) + 1.0
        for x,y in pts:
            px,py = world_to_pixel(float(x), float(y), x0,x1,y0,y1,Wd,Hd)
            c.begin_path()
            c.move_to(px-s,py-s)
            c.line_to(px+s,py+s)
            c.move_to(px-s,py+s)
            c.line_to(px+s,py-s)
            c.stroke()

def axis_and_ticks(x0, x1, y0, y1, nx=7, ny=7, xpad=0.10, ypad=0.15, xfmt=float, yfmt=float):
    if x0==x1: x0,x1 = x0-1.0, x1+1.0
    if y0==y1: y0,y1 = y0-1.0, y1+1.0
    dx,dy = x1-x0, y1-y0
    x0,x1 = x0-xpad*dx, x1+xpad*dx
    y0,y1 = y0-ypad*dy, y1+ypad*dy
    xv = np.linspace(x0, x1, nx)
    yv = np.linspace(y0, y1, ny)
    xt = [((float(v)-x0)/(x1-x0), xfmt(v)) for v in xv]
    yt = [((float(v)-y0)/(y1-y0), yfmt(v)) for v in yv]
    return x0,x1,y0,y1, xt, yt

def to_tensors(xy):
    a = np.array(xy, float)
    return jnp.array(a[:,0:1], dtype=jnp.float32), jnp.array(a[:,1:2], dtype=jnp.float32)


def compute_curve(model_state, is_clf=False, stops=None, two_way=True):
    model, params = model_state
    yp = model.apply(params, xs)
    if is_clf:
        yp = symlog.bin_decode(jax.nn.softmax(yp), stops, two_way)
    return np.array(xs).reshape(-1), np.array(yp).reshape(-1)

def current_curve():
    i = state.get("ckpt_idx", None)
    if i is not None and 0 <= i < len(state["curves"]):
        return state["curves"][i]
    
    # Use the final curve from the training loop instead of improperly re-evaluating
    if state["curves"]:
        return state["curves"][-1]
    
    return None

def draw_points_and_curve():
    x0,x1,y0,y1 = bounds()
    with hold_canvas(plot):
        draw_frame(plot, "click to add points", "x", "y")
        add_markers(plot, train_xy, x0,x1,y0,y1, marker="circle", r=3, color="#000", filled=True, line_width=1.5)
        add_markers(plot, val_xy, x0,x1,y0,y1, marker="cross", r=3, color="#000", filled=False, line_width=1.5)
        curves = current_curve()
        if curves is not None:
            
            for i_curve, curve in enumerate(curves):
                xs, yp = curve
                add_polyline(plot, xs, yp, x0,x1,y0,y1, line_width=2, stroke=distinct_colors[i_curve])

def draw_loss():
    with hold_canvas(loss):

        steps = np.array(state["steps"], float)
        if len(steps) == 0 or len(state["vals"]) == 0:
            draw_frame(loss, "validation loss", "step", "mse")
            return

        # shape: [n_steps, n_models]
        vals = np.array(state["vals"], float)

        # transpose -> [n_models, n_steps]
        vals = vals.T
        
        labels = state.get("run_labels", ["unknown"])

        xmin, xmax = steps.min(), steps.max()
        ymin, ymax = float("inf"), -float("inf")

        # find global y range across all models
        for ys in vals:
            m = np.isfinite(ys)
            if not np.any(m):
                continue
            ymin = min(ymin, ys[m].min())
            ymax = max(ymax, ys[m].max())

        if not np.isfinite(ymin):
            ymin, ymax = 0.0, 1.0

        x0, x1, y0, y1, xt, yt = axis_and_ticks(
            xmin, xmax,
            ymin, ymax,
            nx=7, ny=7
        )

        state["loss_axes"] = (x0, x1, y0, y1)

        draw_frame(loss, "validation loss", "step", "mse")
        draw_ticks(loss, xt, yt)

        # draw one line per model
        for i_model, ys in enumerate(vals):
            add_polyline(
                loss,
                steps,
                ys,
                x0, x1, y0, y1,
                line_width=2,
                stroke=distinct_colors[i_model % len(distinct_colors)]
            )

        legend_x = loss.width - pad - 120
        legend_y = pad + 10
        line_h = 16

        loss.font = "12px sans-serif"

        for i_model, label in enumerate(labels):
            y = legend_y + i_model * line_h
            
            # color line sample
            loss.stroke_style = distinct_colors[i_model % len(distinct_colors)]
            loss.line_width = 2
            loss.begin_path()
            loss.move_to(legend_x, y)
            loss.line_to(legend_x + 20, y)
            loss.stroke()
            
            # text
            loss.fill_style = "#000"
            loss.fill_text(label, legend_x + 28, y + 4)

def draw_hist(canv, history_key, title):
    with hold_canvas(canv):
        pts = [(float(ww), float(vv)) 
               for (ww, vv) in state[history_key] 
               if np.isfinite(vv)]

        draw_frame(canv, title, "w", "mse")

        if not pts:
            return

        xs = np.array([p[0] for p in pts], float)
        ys = np.array([p[1] for p in pts], float)

        xmin, xmax = xs.min(), xs.max()
        ymin, ymax = ys.min(), ys.max()

        x0, x1, y0, y1, xt, yt = axis_and_ticks(
            xmin, xmax,
            ymin, ymax,
            nx=7, ny=7,
            xfmt=int,
            yfmt=float
        )

        draw_ticks(canv, xt, yt)

        # Draw individual model points
        add_markers(
            canv, pts,
            x0, x1, y0, y1,
            marker="circle",
            r=3,
            color="#000",
            filled=True,
            line_width=1.5
        )

        # Calculate averages per width
        groups = defaultdict(list)
        for w_val, loss_val in pts:
            groups[w_val].append(loss_val)
        
        avg_pts = [(w_val, sum(ls)/len(ls)) for w_val, ls in groups.items()]
        
        # Overlay red crosses for averages
        add_markers(
            canv, avg_pts,
            x0, x1, y0, y1,
            marker="cross",
            r=5, 
            color="#FF0000",
            filled=False,
            line_width=2.0
        )

def redraw():
    draw_points_and_curve()
    draw_loss()
    draw_hist(thist, "hist_w_train", "history (w vs last train mse)")
    draw_hist(vhist, "hist_w_val", "history (w vs last val mse)")

def gen_from_expr(_):
    n = int(npts.value)
    p = float(vfrac.value)
    x = np.linspace(-1.0, 1.0, n, endpoint=True).astype(np.float32)
    g = {"np": np}
    y = eval(str(expr.value), g, {"x": x})
    y = np.asarray(y, dtype=np.float32).reshape(-1)
    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]
    n = x.size
    if n == 0:
        status.value="<b style='color:#b00'>expression produced no finite points</b>"
        return
    idx = np.random.permutation(n)
    nv = int(np.clip(round(p*n), 0, n))
    iv, it = idx[:nv], idx[nv:]
    train_xy[:] = list(zip(x[it].tolist(), y[it].tolist()))
    val_xy[:] = list(zip(x[iv].tolist(), y[iv].tolist()))
    state["xlim"][:] = [-1.0, 1.0]
    y0, y1 = float(np.min(y)), float(np.max(y))
    if y0 == y1: y0, y1 = y0-1.0, y1+1.0
    dy = y1 - y0
    state["ylim"][:] = [y0-0.15*dy, y1+0.15*dy]
    state["model"]=None
    state["steps"].clear()
    state["hist_w_val"].clear()
    state["hist_w_train"].clear()
    state["trains"].clear()
    state["vals"].clear()
    state["curves"].clear()
    state["ckpt_idx"]=None
    state["loss_axes"]=None
    status.value=f"generated n={n} train={len(train_xy)} val={len(val_xy)}"
    redraw()

def on_plot_click(x,y):
    x0,x1,y0,y1 = bounds()
    Wd,Hd = plot.width, plot.height
    if not in_axes(x,y,plot):
        return
    dx,dy = pixel_to_world(x,y,x0,x1,y0,y1,Wd,Hd)
    maybe_expand_limits(dx,dy)
    (train_xy if state["mode"]=="train" else val_xy).append((dx,dy))
    redraw()

def on_loss_click(x, y):
    ax = state.get("loss_axes", None)
    if ax is None:
        return
    if not in_axes(x, y, loss):
        return

    x0, x1, y0, y1 = ax
    Wd, Hd = loss.width, loss.height
    sx, _ = pixel_to_world(x, y, x0, x1, y0, y1, Wd, Hd)

    best, bi = None, None

    for i, step in enumerate(state["steps"]):
        d = abs(step - sx)

        if best is None or d < best:
            best, bi = d, i

    if bi is None:
        return

    state["ckpt_idx"] = bi

    step = state["steps"][bi]
    status.value = f"checkpoint step={step}"

    redraw()

def stop(_):...
def undo(_):
    if state["mode"]=="train" and train_xy: train_xy.pop()
    if state["mode"]=="val" and val_xy: val_xy.pop()
    redraw()

def clear(_):
    state["model"]=None
    state["steps"].clear()
    status.value=""
    state["hist_w_val"].clear()
    state["hist_w_train"].clear()
    state["trains"].clear()
    state["vals"].clear()
    state["curves"].clear()
    state["ckpt_idx"]=None
    state["loss_axes"]=None
    redraw()

distinct_colors = [
    "#000000",  # Black
    "#FF0000",  # Bright Red
    "#0000FF",  # Electric Blue
    "#008000",  # Green
    "#FF00FF",  # Magenta / Fuchsia
    "#FF8C00",  # Dark Orange
    "#8B4513",  # Saddle Brown
    "#00CED1",  # Dark Turquoise (High contrast Cyan)
    "#4B0082",  # Indigo
    "#708090"   # Slate Gray
]


def get_means(grads):
    w_grads, b_grads = [], []
    for path, g in jax.tree.leaves_with_path(grads):
        if any('kernel' in str(p) for p in path):
            w_grads.append(g.ravel())
        elif any('bias' in str(p) for p in path):
            b_grads.append(g.ravel())
            
    mean_w = jnp.concatenate(w_grads).mean() if w_grads else 0.0
    mean_b = jnp.concatenate(b_grads).mean() if b_grads else 0.0
    return mean_w, mean_b

x0,x1,_,_ = bounds()
xs = jnp.array(np.linspace(x0, x1, 200, dtype=np.float32).reshape(-1,1))

def train(_):
    if not train_xy:
        status.value="<b style='color:#b00'>add some train points</b>"
        return
    state["stop"]=False
    status.value="training…"

    ws = [int(x) for x in w.value.split(",")]
    archs = [a.strip() for a in arch_type.value.split(",")]
    runs = [(ww, aa) for aa in archs for ww in ws]
    state["run_labels"] = [f"{aa}-{ww}" for ww, aa in runs]

    n_experts = int(e.value)

    xtr, ytr = to_tensors(train_xy)
    xva, yva = (to_tensors(val_xy) if val_xy else (None, None))

    is_clf = task_type.value == "classification"
    out_dim = 1
    stops = None
    
    if is_clf:
        out_dim = int(n_bins_w.value)
        stops = symlog.bin_stops(ytr.ravel(), out_dim, two_way_w.value)
        ytr_target = symlog.bin_encode(ytr.ravel(), stops, two_way_w.value)
        if xva is not None:
            yva_target = symlog.bin_encode(yva.ravel(), stops, two_way_w.value)
    else:
        ytr_target, yva_target = ytr, yva

    models = []
    params_list = []
    opt_states = []
    update_fns = []
    eval_fns = []

    seed = random.randint(0, jnp.iinfo(jnp.int32).max)
    rng = jax.random.key(seed)

    numparams = lambda n: f"{n/1e6:.1f}M" if n >= 1e6 else f"{n/1e3:.1f}k" if n >= 1e3 else str(n)

    for i_run, (one_w, one_arch) in enumerate(runs):

        if one_arch == "le":
            model = make_model_le(one_w, n_experts, out_dim)
        elif one_arch == "de":
            model = make_model_de(one_w, n_experts, out_dim)
        elif one_arch == "mlp":
            model = make_model_mlp(one_w, n_experts, out_dim)
        else:
            raise ValueError(one_arch)

        rng, init_rng = jax.random.split(rng)
        params = model.init(init_rng, xtr)

        total_params = sum(x.size for x in jax.tree.leaves(params))
        state["run_labels"][i_run] += f"-{numparams(total_params)}"
        
        lrval = float(lr.value)
        tx = optax.chain(optax.clip_by_global_norm(1.0), optax.adam(lrval))
        opt_state = tx.init(params)
        
        # JIT compile update and evaluate steps per model to optimize the loop
        @jax.jit
        def update(p, opt_st, x, y, m=model, t=tx):
            def loss_fn(weights):
                preds = m.apply(weights, x)
                if is_clf:
                    return optax.softmax_cross_entropy(preds, y).mean()
                return jnp.mean((preds - y) ** 2)
                
            loss, grads = jax.value_and_grad(loss_fn)(p)
            updates, new_opt_st = t.update(grads, opt_st, p)
            return optax.apply_updates(p, updates), new_opt_st, loss, grads
        
        @jax.jit
        def evaluate(p, x, y_targ, m=model):
            preds = m.apply(p, x)
            if is_clf:
                return optax.softmax_cross_entropy(preds, y_targ).mean()
            return jnp.mean((preds - y_targ)**2)

        models.append(model)
        params_list.append(params)
        opt_states.append(opt_state)
        update_fns.append(update)
        eval_fns.append(evaluate)

    # Store model/param tuples so current_curve() can unpack them seamlessly
    state["model"] = list(zip(models, params_list))
    state["steps"].clear()
    state["vals"].clear()
    state["curves"].clear()
    state["ckpt_idx"] = None
    state["loss_axes"] = None
    redraw()
    
    TT = int(T.value)
    UU = int(upd.value)
    
    for t in range(1, TT+1):
        if state["stop"]:
            break

        save_step = (t==1 or t%UU==0 or t==TT)
        vals = []
        trains = []
        curves = []
        
        for i in range(len(runs)):
            p, opt_st, loss_tr, grads = update_fns[i](params_list[i], opt_states[i], xtr, ytr_target)
            
            params_list[i] = p
            opt_states[i] = opt_st
            
            if save_step:
                if xva is not None:
                    v = float(eval_fns[i](p, xva, yva_target))
                else:
                    v = float("nan")
                vals.append(v)
                trains.append(float(loss_tr))
                curves.append(compute_curve((models[i], p), is_clf, stops, two_way_w.value))
        
        if save_step:
            state["model"] = list(zip(models, params_list))
            state["steps"].append(t)
            state["trains"].append(trains)
            state["vals"].append(vals)
            state["curves"].append(curves)
            redraw()

    lastvs = state["vals"][-1] if state["vals"] else ([float("nan")] * len(runs))
    lastts = state["trains"][-1] if state["trains"] else ([float("nan")] * len(runs))
    
    for (one_w, one_arch), lastt, lastv in zip(runs, lastts, lastvs, strict=True):
        state["hist_w_val"].append((one_w, lastv))
        state["hist_w_train"].append((one_w, lastt))

    redraw()
    status.value = f"{state['run_labels']} finished"
def on_mode_change(_):
    state["mode"] = mode.value

mode.observe(on_mode_change, names="value")
plot.on_mouse_down(lambda x,y: on_plot_click(x,y))
loss.on_mouse_down(lambda x,y: on_loss_click(x,y))

ValueError: mlp.de

TypeError: expected a sequence of integers or a single integer, got '0.2'

TypeError: expected a sequence of integers or a single integer, got '0.2'

TypeError: normal() got multiple values for keyword argument 'loc'

In [9]:
mode.observe(lambda ch: state.__setitem__("mode", ch["new"]), names="value")
train_btn.on_click(train)
stop_btn.on_click(stop)
undo_btn.on_click(undo)
clear_btn.on_click(clear)
expr_btn.on_click(gen_from_expr)

ui = W.VBox([
    W.HBox([mode, undo_btn, clear_btn, train_btn, stop_btn, status]),
    W.HBox([w, e, T, lr, upd]),
    W.HBox([task_type, arch_type, n_bins_w, two_way_w]),
    W.HBox([expr, npts, vfrac, expr_btn]),
    W.HBox([plot, loss]),
    W.HBox([thist, vhist]),
])

display(ui)
redraw()

In [12]:
np.random.normal(3, loc=0, scale=0.1)

TypeError: normal() got multiple values for keyword argument 'loc'

In [10]:
train(None)

ValueError: mlp.de

In [33]:
# Le
ne = 10
wdim = 32
id = 1
od = 1

((id * wdim) + wdim + (wdim * od) + od + (wdim * ne) + ne) * ne

4270

In [ ]:
((id * wdim + wdim + wdim * od) * ne + od)

961

In [ ]:
od * ne + (wdim * ne) * ne + ne * ne - (od) = X 

In [32]:
_params = LeMlp(ne, (od,), (ne,), (wdim,)).init(jax.random.key(123), jnp.zeros((1, 1)))["params"]
_params["ensemble"]["out_layers_0"]["kernel"].shape

(10, 32, 1)

In [9]:
import ipywidgets, ipycanvas
print("ipywidgets:", ipywidgets.__version__)
print("ipycanvas:", ipycanvas.__version__)

ipywidgets: 8.1.8
ipycanvas: 0.14.3


In [1]:
import jax
import jax.numpy as jnp

e = jnp.array([1, 2, 3, 1])
jax.scipy.special.entr(e / e.sum())

Array([0.27798718, 0.35793227, 0.36312765, 0.27798718], dtype=float32)